# Bird Search — Pinecone Full-Text Search

This notebook demonstrates [Pinecone Full-Text Search](https://docs.pinecone.io/guides/search/full-text-search) using a corpus of North American bird Wikipedia articles — one document per bird.

> ⚠️ This feature is in [public preview](https://docs.pinecone.io/guides/search/full-text-search#public-preview). APIs may continue to evolve before general availability.

**Index fields:**

| Field | Type | Query type | Configuration |
|---|---|---|---|
| `bird_name` | string | full text | ✅ `{language: "en"}` |
| `intro` | string | full text | ✅ `{language: "en"}` |
| `body` | string | full text | ✅ `{language: "en", stemming: True}` |
| `image_embedding` | dense vector | semantic | dim 768, cosine |

## Prerequisites

- **Pinecone account and API key** — [sign up](https://app.pinecone.io/) and create an API key in the console.
- **Google Cloud project with Gemini API access** — enable the Gemini API and create an API key in [Google AI Studio](https://aistudio.google.com/apikey). Required for section 2 (image embedding) and section 16 (vector search). This notebook uses `gemini-embedding-2` embedding model for multi-modal embedding.

## Contents

- [0. Install the SDK](#0-install-the-sdk)
- [1. Connect to the index](#1-connect-to-the-index)
- [2. Create index and load data](#2-create-index-and-load-data)
- [3. Display helper](#3-display-helper)
- [How tokens work](#how-tokens-work)
- [4. Single-term token match — `"migration"`](#4-single-term-token-match-migration)
- [5. Searching `bird_name` — and blending multiple fields](#5-searching-bird-name-and-blending-multiple-fields)
- [6. Introducing Lucene syntax: single-term `query_string`](#6-introducing-lucene-syntax-single-term-query-string)
- [7. Requiring both terms: AND operator](#7-requiring-both-terms-and-operator)
- [8. Excluding terms: NOT operator](#8-excluding-terms-not-operator)
- [9. Exact phrase vs. token OR in `bird_name`](#9-exact-phrase-vs-token-or-in-bird-name)
- [10. Phrase proximity (slop): flexible phrase matching in `body`](#10-phrase-proximity-slop-flexible-phrase-matching-in-body)
- [11. Boosting: influencing ranking by term importance](#11-boosting-influencing-ranking-by-term-importance)
- [12. Cross-field `query_string`: multiple fields in one clause](#12-cross-field-query-string-multiple-fields-in-one-clause)
- [13. Combining everything: a production-grade query](#13-combining-everything-a-production-grade-query)
- [14. Regex: token-level pattern matching](#14-regex-token-level-pattern-matching)
- [15. Phrase prefix: autocomplete](#15-phrase-prefix-autocomplete)
- [16. Dense vector ranking with a phrase-match filter](#16-dense-vector-ranking-with-a-phrase-match-filter)
- [Summary](#summary)
- [17. Cleanup](#17-cleanup)

## 0. Install the SDK

> ⚠️ Because this feature is in public preview, it uses the `preview` namespace. `pc.preview.*` targets Pinecone API version `2026-01.alpha`. Pin your SDK version when relying on it.

In [24]:
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

%pip install --quiet \
    "pinecone==9.0.0" \
    "pinecone-notebooks==0.1.1" \
    "httpx[http2]==0.28.1" \
    "msgspec==0.21.1" \
    "orjson==3.11.8" \
    "tqdm==4.67.3" \
    "google-genai==1.73.1" \
    "python-dotenv==1.2.2" \
    "pillow==12.2.0"

Note: you may need to restart the kernel to use updated packages.


## 1. Connect to the index

You will need a free Pinecone API key. If you're running in a Colab environment, the code below will either help you sign up for a new Pinecone account or authenticate you. Then it will create a new API key and set it as an environment variable. If you are not running in a Colab environment, it will use your environment variables or prompt you to enter the API key and then set it in the environment.

### Setup Pinecone API key

In [ ]:
import os
from getpass import getpass

import pinecone
from dotenv import load_dotenv

load_dotenv()  # loads .env into os.environ; no-op if file absent


def get_pinecone_api_key():
    """Get Pinecone API key from environment, Colab auth, or prompt.

    Only necessary for notebooks. When using Pinecone yourself,
    you can use environment variables or the like to set your API key.
    """
    api_key = os.environ.get("PINECONE_API_KEY")

    if api_key is None:
        try:
            from pinecone_notebooks.colab import Authenticate

            Authenticate()
            api_key = os.environ.get("PINECONE_API_KEY")
        except ImportError:
            print("Pinecone API key not found in environment.")
            api_key = getpass("Please enter your Pinecone API key: ")
            os.environ["PINECONE_API_KEY"] = api_key

    return api_key


PINECONE_API_KEY = get_pinecone_api_key()

### Setup Google API key

In [ ]:
def get_google_api_key():
    """Get Google API key from environment or prompt."""
    api_key = os.environ.get("GOOGLE_API_KEY")

    if api_key is None:
        print("Google API key not found in environment.")
        api_key = getpass("Please enter your Google API key: ")
        os.environ["GOOGLE_API_KEY"] = api_key

    return api_key


GOOGLE_API_KEY = get_google_api_key()

### Initialize the client and connect

In [ ]:
pc = pinecone.Pinecone(
    api_key=PINECONE_API_KEY,
    source_tag="pinecone_examples:docs:full_text_search",
)

INDEX_NAME = "bird-search-fts"
NAMESPACE = "birds"  # namespaces partition a single index — useful for isolating datasets or tenants

print(f"SDK version : {pinecone.__version__}")
print(f"Running in  : {'Colab' if IN_COLAB else 'local Jupyter'}")

if pc.preview.indexes.exists(INDEX_NAME):
    idx = pc.preview.index(name=INDEX_NAME)
    print(f"Connected to: {INDEX_NAME}")
else:
    print(f"Index '{INDEX_NAME}' not found — run section 2 to create and populate it")

## 2. Create index and load data

### Download bird corpus

First, you'll download the bird corpus and define `DATA_DIR`. Run this even if you've previously loaded data into your index as it sets `DATA_DIR`.

In [ ]:
import json
import pathlib
import time
import urllib.request
import zipfile

if not pathlib.Path("bird-search-example-main").exists():
    print("Downloading bird corpus...")
    urllib.request.urlretrieve(
        "https://github.com/pinecone-io/bird-search-example/archive/refs/heads/main.zip",
        "main.zip",
    )
    with zipfile.ZipFile("main.zip") as zf:
        zf.extractall()
    pathlib.Path("main.zip").unlink()
    print("Done.")

DATA_DIR = (
    pathlib.Path(
        os.environ.get("BIRD_DATA_DIR", "bird-search-example-main/parsed_birds")
    )
    .expanduser()
    .resolve()
)
N_BIRDS = 200
EMBED_DIM = 768
GEMINI_MODEL = "gemini-embedding-2"
print(f"DATA_DIR: {DATA_DIR}")

### Create index and load data

Now, you'll create the index and upsert the data. If section 1 printed **"Connected to: bird-search-fts"**, skip this step — your index is already populated.

In [ ]:
from google import genai as _genai
from google.genai import types as genai_types
from PIL import Image
from pinecone.preview import SchemaBuilder
from tqdm import tqdm

gem_loader = _genai.Client(api_key=GOOGLE_API_KEY)
EMBED_CONFIG = genai_types.EmbedContentConfig(output_dimensionality=EMBED_DIM)

if not pc.preview.indexes.exists(INDEX_NAME):
    schema = (
        SchemaBuilder()
        .add_string_field("bird_name", full_text_search={"language": "en"})
        .add_string_field("intro", full_text_search={"language": "en"})
        .add_string_field("body", full_text_search={"language": "en", "stemming": True})
        .add_dense_vector_field("image_embedding", dimension=EMBED_DIM, metric="cosine")
        .build()
    )
    pc.preview.indexes.create(name=INDEX_NAME, schema=schema)
    print(f"Created index '{INDEX_NAME}' — waiting for Ready...")
    while not pc.preview.indexes.describe(INDEX_NAME).status.ready:
        time.sleep(5)
        print("  still initializing...")
    print("Index is ready.")

idx = pc.preview.index(name=INDEX_NAME)
print(f"Connected to: {INDEX_NAME}")

In [ ]:
# ── Load metadata + filter usable birds ───────────────────────────────────────
meta = json.loads((DATA_DIR / "parsing_metadata.json").read_text())

slugs = [
    slug
    for slug, entry in meta.items()
    if (DATA_DIR / "text" / entry.get("text_file", "")).exists()
    and entry.get("images")
    and (DATA_DIR / "images" / entry["images"][0]["local_path"]).exists()
]
slugs = sorted(slugs)[:N_BIRDS]

print(f"Loading {len(slugs)} birds from {DATA_DIR}")


# ── Helper: split article text into intro + body ───────────────────────────────
def split_intro_body(text):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    if not paragraphs:
        return "", ""
    return paragraphs[0], "\n\n".join(paragraphs[1:])


# ── Embed images + build documents ────────────────────────────────────────────
docs = []
for slug in tqdm(slugs, desc="Embedding images"):
    entry = meta[slug]
    img_path = DATA_DIR / "images" / entry["images"][0]["local_path"]
    with Image.open(img_path) as img:
        img.load()
        resp = gem_loader.models.embed_content(
            model=GEMINI_MODEL,
            contents=img,
            config=EMBED_CONFIG,
        )
    image_embedding = list(resp.embeddings[0].values)

    text = (DATA_DIR / "text" / entry["text_file"]).read_text(encoding="utf-8")
    intro, body = split_intro_body(text)

    docs.append(
        {
            "_id": slug,
            "bird_name": slug.replace("_", " "),
            "intro": intro,
            "body": body,
            "image_embedding": image_embedding,
        }
    )

print(f"Built {len(docs)} documents.")

# ── Upsert ─────────────────────────────────────────────────────────────────────
result = idx.documents.batch_upsert(
    namespace=NAMESPACE,
    documents=docs,
    batch_size=50,
    max_workers=4,
    show_progress=True,
)
print(f"Uploaded {result.successful_item_count} / {result.total_item_count} documents")

# ── Wait until searchable ──────────────────────────────────────────────────────
print("Waiting for documents to be indexed...")
while True:
    probe = idx.documents.search(
        namespace=NAMESPACE,
        top_k=1,
        score_by=[{"type": "text", "field": "body", "query": "bird"}],
        include_fields=[],
    )
    if probe.matches:
        print("Data is searchable — ready to query.")
        break
    time.sleep(5)
    print("  not yet indexed, retrying...")

Note: it is expected to have some failures in the batch upsert as some of the document bodies will exceed the max token count. Expect to see `Uploaded 150 / 200 documents`.

## 3. Display helper

A small function to print search results in a readable format.

In [ ]:
import textwrap
from pathlib import Path


def show_results(response, snippet_field="intro", max_lines=10, image_dir=None):
    """Print search results with score, bird name, and a text snippet. Pass max_lines=-1 for full field."""
    if not response.matches:
        print("(no matches)")
        return
    if image_dir is not None:
        from IPython.display import Image as IPImage
        from IPython.display import display
    for doc in response.matches:
        name = doc.get("bird_name") or doc._id
        print(f"Score {doc.score:.4f}  [{doc._id}]  {name}")
        if image_dir is not None:
            img_path = Path(image_dir) / doc._id / f"{doc._id}_1.jpg"
            if img_path.exists():
                display(IPImage(filename=str(img_path), width=220))
        snippet = doc.get(snippet_field) or ""
        if snippet:
            if max_lines == -1:
                wrapped = textwrap.fill(
                    snippet.strip(),
                    width=88,
                    initial_indent="    ",
                    subsequent_indent="    ",
                )
            else:
                wrapped = textwrap.fill(
                    snippet.strip(),
                    width=88,
                    initial_indent="    ",
                    subsequent_indent="    ",
                    max_lines=max_lines,
                    placeholder=" …",
                )
            print(wrapped)
        print()

## How tokens work

Every full-text-search field runs through an **analyzer pipeline** at index time and again at query time:

1. **Split** on whitespace and punctuation — `state-of-the-art` becomes `state`, `of`, `the`, `art`
2. **Lowercase** every token
3. **Stem** to root form if `stemming: true` — `migrating` → `migrat`, `models` → `model`
4. **Drop stop words** if `stop_words: true` — `the`, `and`, etc.
5. **Cap** each token at 40 characters

Because the same pipeline runs on both sides, a token that scores in BM25 will also match in a `$match_phrase` filter on the same field. In this corpus, `bird_name` and `intro` use the pipeline with stemming **off**; `body` has stemming **on**.

> `dense_vector` fields use a completely different tokenizer internal to the embedding model — those tokens are never visible and `$match_*` filters don't apply to them.

Read more about this [here](https://pinecone-andrew-fts-docs-public-preview.mintlify.app/guides/search/full-text-search#tokens-and-analyzers).

## 4. Single-term token match — `"migration"`

A plain `text` score clause performs a full-text keyword search on a single field.
Here we search the `body` field (stemming enabled) for documents containing the
token *migration* — picking up morphological variants like *migrating*, *migratory*, etc.

In [ ]:
QUERY = "migration"

response = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[
        {"type": "text", "field": "body", "query": QUERY},
    ],
    include_fields=["bird_name", "body"],
)

print(f'Top {len(response.matches)} results for "{QUERY}" in body:\n')
show_results(response, "body", 100)

## 5. Searching `bird_name` — and blending multiple fields

Each `type: "text"` clause targets exactly one field. To search across
fields simultaneously, pass **multiple clauses** in `score_by` — one per field.
A single search request ranks by one scoring method only (BM25 text, Lucene query syntax, dense vector, or sparse vector). Every contributing field within that scoring method weighs equally; there is no per-clause weight parameter.

We'll use the query `"sparrow"` to make the contrast concrete:

- **Single-field on `bird_name`:** finds birds *named* sparrow — only 5 in
  this corpus, all with the English word in their formal name.
- **Blended `bird_name` + `intro` + `body`:** surfaces a fifth bird,
  *Ammospiza maritima mirabilis*, whose stored `bird_name` is the Latin
  binomial. Its article explicitly describes it as a sparrow — the body
  text uses the word six times — but name-only search misses it entirely.

In [ ]:
# ── Single-field search: bird_name only ────────────────────────────────────
QUERY = "sparrow"

response_name = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "text", "field": "bird_name", "query": QUERY}],
    include_fields=["bird_name", "intro"],
)

print(f'Searching bird_name only for "{QUERY}":\n')
show_results(response_name, "intro")

Name-only returns just 5 results — every bird with "sparrow" in its formal
English name. Now blend all three text fields. The blended search scores body
text too, so *Ammospiza maritima mirabilis* (the Cape Sable seaside sparrow)
rises into the top 5 even though its `bird_name` field contains only the Latin
binomial.

> **Stemming note:** `bird_name` and `intro` have stemming **off** — these
> clauses require the literal token `"sparrow"`. `body` has stemming **on**,
> so `"sparrows"` (plural) also matches there without any extra syntax.

In [ ]:
# ── Blended search: bird_name + intro + body ───────────────────────────────
response_blend = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[
        {"type": "text", "field": "bird_name", "query": QUERY},
        {"type": "text", "field": "intro", "query": QUERY},
        {"type": "text", "field": "body", "query": QUERY},
    ],
    include_fields=["bird_name", "intro"],
)

print(f'Blending bird_name + intro + body for "{QUERY}":\n')
show_results(response_blend, "intro")

# Show which IDs appear in blend but not in name-only search
name_ids = {d._id for d in response_name.matches}
blend_ids = {d._id for d in response_blend.matches}
new_in_blend = blend_ids - name_ids
if new_in_blend:
    print("IDs that appear in blended results but not in name-only:")
    for doc in response_blend.matches:
        if doc._id in new_in_blend:
            print(f"  [{doc._id}]  {doc.get('bird_name')}  (score {doc.score:.4f})")
else:
    print("Top-5 results are the same in both queries.")

## 6. Introducing Lucene syntax: single-term `query_string`

`type: "query_string"` exposes the full Lucene query syntax. The field name is
embedded **inside** the query string itself.

For a single term this produces identical results to the `type: "text"`
equivalent — it’s the baseline before we start using operators.

In [ ]:
# ── query_string — single term, equivalent to type:text ────────────────────────
response_qs = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": "body:(migration)"}],
    include_fields=["bird_name", "body"],
)

print("query_string  body:(migration)\n")
show_results(response_qs, "body", 6)

## 7. Requiring both terms: AND operator

`type: "text"` with multiple words uses **OR semantics** — a document only needs
one term. `query_string` lets you enforce **AND**, so both terms must appear
in the field.

The `+` prefix is equivalent: `body:(+aquatic +diving)` requires both;
`body:(+aquatic diving)` requires “aquatic” and optionally “diving”.

In [ ]:
# ── OR (type:text) vs AND (query_string) ────────────────────────────────────
QUERY_OR = "aquatic diving"
QUERY_AND = "body:(aquatic AND diving)"

response_or = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "text", "field": "body", "query": QUERY_OR}],
    include_fields=["bird_name", "body"],
)

response_and = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": QUERY_AND}],
    include_fields=["bird_name", "body"],
)

print(f'type:text  "{QUERY_OR}"  (OR — either term)\n')
show_results(response_or, "body", 4)

print(f"query_string  {QUERY_AND}  (AND — both terms required)\n")
show_results(response_and, "body", 4)

or_ids = {d._id for d in response_or.matches}
and_ids = {d._id for d in response_and.matches}
or_only = or_ids - and_ids
if or_only:
    print("In OR results but not AND (likely has only one term):")
    for doc in response_or.matches:
        if doc._id in or_only:
            print(f"  [{doc._id}]  {doc.get('bird_name')}")

## 8. Excluding terms: NOT operator

`NOT` (or the `-` prefix) excludes any document where the term appears in the
field — a **hard filter on field content**, not a proximity constraint. A
document that discusses both raptors and owls is excluded because “owl”
appears somewhere in `body`, even if the article is primarily about a hawk.

In [ ]:
# ── NOT: raptors excluding owls ───────────────────────────────────────────
response_not = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": "body:(raptor NOT owl)"}],
    include_fields=["bird_name", "body"],
)

print("query_string  body:(raptor NOT owl)\n")
show_results(response_not, "body")

## 9. Exact phrase vs. token OR in `bird_name`

The `bird_name` field is short (2–4 words), which makes the difference between
phrase and token-OR immediately visible in results.

- **Token OR** `bird_name:(crested hummingbird)` — matches any name containing
  *crested* **or** *hummingbird*: Antillean crested hummingbird, Black-crested titmouse
- **Exact phrase** `bird_name:("crested hummingbird")` — only names where the tokens
  *crested* and *hummingbird* appear **adjacent and in that order**: Antillean crested hummingbird

Note: `bird_name` has stemming **off**, so phrase matching is purely
token-level with no morphological expansion.

In [ ]:
# ── Token OR vs exact phrase in bird_name ───────────────────────────────────
response_tokens = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": "bird_name:(crested hummingbird)"}],
    include_fields=["bird_name", "intro"],
)

response_phrase = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": 'bird_name:("crested hummingbird")'}],
    include_fields=["bird_name", "intro"],
)

print("Token OR  bird_name:(crested hummingbird)\n")
show_results(response_tokens, "intro")

print('Exact phrase  bird_name:("crested hummingbird")\n')
show_results(response_phrase, "intro")

## 10. Phrase proximity (slop): flexible phrase matching in `body`

A strict phrase requires tokens to be **adjacent**. Adding `~N` (slop)
allows up to *N* intervening or reordered tokens, catching natural-language
paraphrases of the same concept.

`body:("nest colony"~3)` matches:
- “nest in a colony” (2 words between)
- “colonial nesting” (reversed order counts against the slop budget)
- "nest in colonies" (1 word between, stemming on)
- “nesting in large colonies” (2 words between, stemming on)

Contrast with the strict phrase `body:("nest colony")`, which requires the
tokens to be directly adjacent.

In [ ]:
# ── Strict phrase vs. proximity phrase ──────────────────────────────────────
response_strict = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": 'body:("nest colony")'}],
    include_fields=["bird_name", "body"],
)

response_slop = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": 'body:("nest colony"~3)'}],
    include_fields=["bird_name", "body"],
)

print('Strict phrase  body:("nest colony")\n')
show_results(response_strict, "body", 60)

print('Proximity slop  body:("nest colony"~3)\n')
show_results(response_slop, "body", 60)

strict_ids = {d._id for d in response_strict.matches}
slop_ids = {d._id for d in response_slop.matches}
gained = slop_ids - strict_ids
if gained:
    print("Gained by relaxing to slop~3 (not in strict results):")
    for doc in response_slop.matches:
        if doc._id in gained:
            print(f"  [{doc._id}]  {doc.get('bird_name')}")
else:
    print("Same top-5 for both queries.")

## 11. Boosting: influencing ranking by term importance

`^N` multiplies a term’s BM25 score contribution by *N* **without** making it
required. Documents that lack the boosted term can still appear if they score
well on the other terms — boosting shapes the ranking, it doesn’t filter.

```
body:(foraging^3 feeding diet)
```

- `foraging^3` — three times the weight of an unboosted term
- `feeding`, `diet` — unboosted; contribute normally

Phrases can be boosted too: `body:("aerial foraging"^2 insects)` boosts the
exact adjacent phrase rather than a single token.

In [ ]:
# ── Boosting: foraging^3 outweighs feeding / diet ───────────────────────────
response_boosted = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": "body:(foraging^3 feeding diet)"}],
    include_fields=["bird_name", "body"],
)

response_flat = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": "body:(foraging feeding diet)"}],
    include_fields=["bird_name", "body"],
)

print("Boosted  body:(foraging^3 feeding diet)\n")
show_results(response_boosted, "body", 4)

print("Flat (no boost)  body:(foraging feeding diet)\n")
show_results(response_flat, "body", 4)

## 12. Cross-field `query_string`: multiple fields in one clause

A single `query_string` clause can reference **multiple fields** by combining
field-scoped sub-clauses with boolean operators.

```
bird_name:(hawk) AND body:(hunting prey)
```

The top-level `AND` requires **both** sub-clauses to match: a bird must have
“hawk” in its name **and** “hunting” or “prey” in its body.

Contrast with the multi-clause `score_by` approach from step 4: that blends
scores across separate clauses (any clause can match). This enforces a
**cross-field boolean constraint** in a single expression.

In [ ]:
# ── Cross-field AND: hawk in name + hunting/prey in body ────────────────────
response_cross = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[
        {
            "type": "query_string",
            "query": "bird_name:(hawk) AND body:(hunting prey)",
        }
    ],
    include_fields=["bird_name", "body"],
)

print("query_string  bird_name:(hawk) AND body:(hunting prey)\n")
show_results(response_cross, "body")

## 13. Combining everything: a production-grade query

Each of the preceding steps introduced one concept. This query composes them
all into a single expression:

```
bird_name:(hawk^2 OR eagle) AND
body:(("dense vegetation" OR "forest canopy") AND hunt -fish)
```

Clause by clause:

| Clause | Concept |
|---|---|
| `bird_name:(hawk^2 OR eagle)` | boost (step 10) + token OR on `bird_name` (step 8) |
| `AND` | cross-field boolean (step 11) |
| `"dense vegetation" OR "forest canopy"` | exact phrases targeting forest-interior hunters (step 8) |
| `AND hunt` | require term — AND operator (step 6) |
| `-fish` | exclude term — NOT operator (step 7) |

The intent is **forest-interior hawks and eagles that actively hunt, excluding
piscivorous species**. Two exclusions work at different levels:

- **Name-level:** using `eagle` instead of `falcon` keeps open-country falcons
  like the Aplomado out of the candidate set entirely.
- **Body-level:** `-fish` hard-filters the Black-collared hawk, whose diet
  is described as "mainly composed of fish".

Expected top results: the Bicolored hawk ("flying through dense vegetation to
ambush unsuspecting prey") and the Black-and-white hawk-eagle ("nests in the
forest canopy").

> **Stemming:** `body` has stemming on, so `hunt` also matches *hunting* and
> *hunted* — no extra syntax needed.
>
> **Operator precedence:** AND binds tighter than OR. Use explicit
> parentheses when mixing to avoid surprises.

In [ ]:
# ── Composed query: forest hawks/eagles that hunt, no fish-eating specialists ─
COMPOSED = (
    "bird_name:(hawk^2 OR eagle) AND "
    'body:(("dense vegetation" OR "forest canopy") AND hunt -fish)'
)

response_composed = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": COMPOSED}],
    include_fields=["bird_name", "body"],
)

print(f"query_string  {COMPOSED}\n")
show_results(response_composed, "body")

## 14. Regex: token-level pattern matching

Lucene `/pattern/` regex syntax works inside `query_string` clauses. Wrap the
pattern in forward slashes inside the field clause:

```
bird_name:/.*bird/
```

**Key constraint:** the pattern matches the **entire indexed token**, not the
full field string. `bird_name:/.*bird/` asks — for each token in `bird_name`,
does it match `.*bird` end-to-end?

This unlocks suffix matching that no other query type supports. A simple
token search for `bird_name:(bird)` only finds documents where "bird" is a
standalone token — no bird in this corpus has that. Regex finds every compound
word that ends in "bird": `hummingbird`, `mockingbird`, `puffbird`, etc.

> **Stemming note:** `bird_name` has stemming **off**, so regex operates on
> the raw lowercased tokens — no morphological expansion.

In [ ]:
# ── Token search: "bird" as a standalone token ───────────────────────────────
# No bird in this corpus has "bird" as a separate token in its name —
# hummingbird, mockingbird, puffbird, etc. are all single compound tokens.
response_token = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": "bird_name:(bird)"}],
    include_fields=["bird_name"],
)

print("token  bird_name:(bird)\n")
show_results(response_token, "bird_name")

# ── Regex suffix: every token ending in "bird" ────────────────────────────────
# The pattern /.*bird/ matches the full token end-to-end, so it catches any
# compound word whose final characters are "bird".
response_regex = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[{"type": "query_string", "query": "bird_name:/.*bird/"}],
    include_fields=["bird_name"],
)

print("regex  bird_name:/.*bird/\n")
show_results(response_regex, "bird_name")

token_ids = {d._id for d in response_token.matches}
regex_ids = {d._id for d in response_regex.matches}
gained = regex_ids - token_ids
if gained:
    print("Found by regex but not by token search (compound tokens ending in 'bird'):")
    for doc in response_regex.matches:
        if doc._id in gained:
            print(f"  [{doc._id}]  {doc.get('bird_name')}")

## 15. Phrase prefix: autocomplete

Appending `*` after a quoted phrase treats the **last token as a prefix**.
All preceding tokens must match exactly and adjacently; only the final token
is expanded.

`body:("tropcial fo"*)` matches:
- "tropical forest"
- "tropics. Foraging" (when stemming is on)

But not:
- "tropical rain forest"

> **Constraint:** single-term prefix wildcards (`tropic*`) are not supported —
> the phrase must contain at least two tokens before the `*`.
>
> **Practical use:** power an autocomplete UI widget by passing the partial
> query string directly into this pattern as the user types.

In [ ]:
# ── Phrase prefix: autocomplete ──────────────────────────────────────────────
response_prefix = idx.documents.search(
    namespace=NAMESPACE,
    top_k=2,
    score_by=[{"type": "query_string", "query": 'body:("tropical fo"*)'}],
    include_fields=["bird_name", "body"],
)

print('query_string  body:("tropical fo"*)\n')
show_results(response_prefix, "body", -1)

## 16. Dense vector ranking with a phrase-match filter

Only one scoring method is active at a time — `dense_vector` and
`text`/`query_string` clauses cannot be mixed in a single `score_by` array.
Both retrieval methods can still be combined in one query: full-text keyword
matching applies as a hard filter **before** scoring. The
scoring method (in this case `dense_vector`) orders whatever remains after filtering. Filtering guarantees the
results mention the right terms; vectors rank by semantic or visual similarity
within that constrained set.

The query vector is produced by embedding a natural-language description with
Gemini's multimodal embedding model at the same dimensionality (768) as the
stored `image_embedding` vectors. Gemini embeds text and images into the same
space, so a text description of a visual concept lands near images of that concept.

Three text-match filter operators:

| Operator | Semantics |
|---|---|
| `$match_phrase` | tokens adjacent and in order (exact phrase) |
| `$match_all` | all tokens present, any order |
| `$match_any` | at least one token present |

In [ ]:
from google import genai
from google.genai import types

gem = genai.Client(api_key=GOOGLE_API_KEY)
EMBED = types.EmbedContentConfig(output_dimensionality=768)

VISUAL_QUERY = "red bird with a prominent crest"

query_vector = (
    gem.models.embed_content(
        model="gemini-embedding-2",
        contents=VISUAL_QUERY,
        config=EMBED,
    )
    .embeddings[0]
    .values
)

print(f'Query : "{VISUAL_QUERY}"')
print(f"Vector: {len(query_vector)}-dim  first 4 values: {list(query_vector[:4])}")

In [ ]:
# ── Dense vector ranked by image similarity, filtered to exact phrase ────────
# filter applies BEFORE scoring — only documents whose body contains
# the exact adjacent phrase "prominent crest" are eligible to be ranked.
response_phrase = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[
        {
            "type": "dense_vector",
            "field": "image_embedding",
            "values": query_vector,
        }
    ],
    filter={"body": {"$match_phrase": "prominent crest"}},
    include_fields=["bird_name", "body"],
)

print('dense_vector image_embedding  filter: body $match_phrase "prominent crest"\n')
show_results(response_phrase, "body", -1, image_dir=DATA_DIR / "images")

In [ ]:
# ── $match_all: both tokens present anywhere in body (any order) ─────────────
response_all = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[
        {
            "type": "dense_vector",
            "field": "image_embedding",
            "values": query_vector,
        }
    ],
    filter={"body": {"$match_all": "prominent crest"}},
    include_fields=["bird_name", "body"],
)

print('dense_vector image_embedding  filter: body $match_all "prominent crest"\n')
show_results(response_all, "body", -1, image_dir=DATA_DIR / "images")

# $match_all is broader than $match_phrase — both tokens must appear but
# need not be adjacent. Expect more results or a different ranking.
phrase_ids = {d._id for d in response_phrase.matches}
all_ids = {d._id for d in response_all.matches}
gained = all_ids - phrase_ids
if gained:
    print("In $match_all but not $match_phrase (tokens present but not adjacent):")
    for doc in response_all.matches:
        if doc._id in gained:
            print(f"  [{doc._id}]  {doc.get('bird_name')}")

In [ ]:
# ── Compound filter: phrase in body AND token match in intro ─────────────────
# Logical operators ($and, $or) compose filter clauses across fields.
response_compound = idx.documents.search(
    namespace=NAMESPACE,
    top_k=5,
    score_by=[
        {
            "type": "dense_vector",
            "field": "image_embedding",
            "values": query_vector,
        }
    ],
    filter={
        "$and": [
            {"body": {"$match_phrase": "woody woodpecker"}},
            {"body": {"$match_any": "prominent crest"}},
        ]
    },
    include_fields=["bird_name", "intro", "body"],
)

print('dense_vector  filter: body $match_phrase "woody woodpecker"')
print('              AND body $match_any "prominent crest"\n')
show_results(response_compound, "body", -1, image_dir=DATA_DIR / "images")

## 17. Cleanup

Delete the index and all its data when you're done exploring. This is
irreversible — re-run section 2 to recreate and repopulate it.

In [ ]:
pc.preview.indexes.delete(INDEX_NAME)
print(f"Deleted index '{INDEX_NAME}'")

## Summary

This notebook covered Pinecone Full-Text Search end-to-end:

- **Token and field search** — `type:text` and `type:query_string` for single-term, multi-term OR/AND/NOT, and cross-field queries
- **Lucene query syntax** — exact phrases, proximity slop, boosting, regex, and phrase prefix
- **Hybrid ranking** — combining dense vector similarity with text-match filters (`$match_phrase`, `$match_all`, `$match_any`)

**Next steps:**

- [Full-text search guide](https://docs.pinecone.io/guides/search/full-text-search) — full API reference and supported Lucene syntax
- [Bird Search web app](https://github.com/pinecone-io/bird-search-v2) - a Streamlit app to run in the browser